# AeroPulse PM2.5 — 01A: Historical Coverage Audit

Measure exactly what the current dataset can and cannot support, so that Phase-7 results
are read against the data behind them rather than against the data we would like.

## Objective

The roadmap's Sprint 1 asks for a **3–5 year** CPCB-primary canonical dataset. What exists
is roughly **19 months** of OpenAQ observations. That gap is not a detail — it determines
which conclusions from notebooks 06–08 are trustworthy and which are single-instance
anecdotes.

This notebook is read-only. It measures coverage and states, per analysis, whether the
data supports it. It does not fetch, fix or fabricate anything.

### Why the gap exists

OpenAQ's Indian PM2.5 archive has a real hole. Major CPCB/DPCC stations (R K Puram, Anand
Vihar, Punjabi Bagh) hold data for 2016-02 to 2018-02 and then nothing until 2025-02, and
the median Indian sensor's first observation is 2025-02-18. So no OpenAQ-only pipeline can
produce a multi-year Indian history today, regardless of how it is written.

CPCB CAAQMS is the authoritative source for the missing period. It has **no public
historical API** — `app.cpcbccr.com` is a captcha-gated JavaScript portal — so it cannot
be automated. `01B_cpcb_pm25_ingestion.ipynb` provides the local-drop path for
portal exports instead.

In [1]:
# ============================================================================
# COVERAGE AUDIT - read-only
# ============================================================================
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

_dotenv = find_dotenv(usecwd=True)
if _dotenv:
    load_dotenv(_dotenv, override=True)

PROJECT_ROOT = Path(os.getenv("AEROPULSE_PROJECT_ROOT", ".")).resolve()
BASE_FILE = PROJECT_ROOT / "data" / "pm25" / "processed" / "base" / "base_dataset.parquet"
OUT_DIR = PROJECT_ROOT / "artifacts" / "pm25" / "phase7"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# What the roadmap asks for, so the audit compares against a stated target
# rather than against a feeling.
ROADMAP_TARGET = {
    "min_years": 3,
    "preferred_years": 5,
    "min_winter_cycles": 3,
    "min_post_monsoon_cycles": 3,
    "rationale": "seasonal generalisation cannot be asserted from one instance of "
                 "a season; a model tuned on one winter is tuned on that winter",
}

if not BASE_FILE.exists():
    raise FileNotFoundError(f"{BASE_FILE} not found. Run Phase 1 first.")

df = pd.read_parquet(BASE_FILE, columns=["timestamp_utc", "location_id",
                                         "station_name", "lat", "lon", "pm25",
                                         "source"])
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df["local"] = df["timestamp_utc"].dt.tz_convert("Asia/Kolkata")
df["month"] = df["local"].dt.month
df["season"] = np.select(
    [df.month.isin([10, 11]), df.month.isin([12, 1, 2]), df.month.isin([3, 4, 5])],
    ["post_monsoon", "winter", "pre_monsoon"], default="monsoon")

span_days = (df.timestamp_utc.max() - df.timestamp_utc.min()).days
span_years = span_days / 365.25
print(f"Rows        : {len(df):,}")
print(f"Stations    : {df.location_id.nunique()}")
print(f"Sources     : {df.source.value_counts().to_dict()}")
print(f"Window      : {df.timestamp_utc.min().date()} -> {df.timestamp_utc.max().date()}")
print(f"Span        : {span_days} days = {span_years:.2f} years")
print(f"\nRoadmap target: {ROADMAP_TARGET['min_years']}-"
      f"{ROADMAP_TARGET['preferred_years']} years")
print(f"Shortfall     : {ROADMAP_TARGET['min_years'] - span_years:.2f} years "
      f"({span_years / ROADMAP_TARGET['min_years']:.0%} of the minimum)")

Rows        : 1,705,252
Stations    : 149
Sources     : {'OpenAQ': 1705252}
Window      : 2025-02-18 -> 2026-09-09
Span        : 567 days = 1.55 years

Roadmap target: 3-5 years
Shortfall     : 1.45 years (52% of the minimum)


### Seasonal cycles, which is the number that actually binds

Total span is the wrong summary. A model is asked to generalise across *seasons*, so what
matters is how many distinct instances of each season it has seen. One winter means every
winter conclusion rests on a single sample, and no amount of cross-validation inside that
winter creates a second one.

In [2]:
# ============================================================================
# SEASONAL CYCLE COUNT
# ============================================================================
# A "cycle" is one contiguous occurrence of a season. Counting distinct
# (year, season) pairs would overcount winter, which straddles the new year.
df["season_year"] = np.where(df.month.isin([1, 2]), df.local.dt.year - 1,
                             df.local.dt.year)
cycles = (df.groupby(["season", "season_year"])
            .agg(rows=("pm25", "size"), days=("local", lambda s: s.dt.date.nunique()),
                 mean_pm25=("pm25", "mean"), max_pm25=("pm25", "max"),
                 extreme_rows=("pm25", lambda s: int((s >= 150).sum())))
            .reset_index())
# A handful of rows at the edge of the window is not a cycle.
cycles["is_substantial"] = cycles.days >= 30
display(cycles.round(1))

counts = (cycles[cycles.is_substantial].groupby("season").size()
          .reindex(["winter", "pre_monsoon", "monsoon", "post_monsoon"])
          .fillna(0).astype(int))
print("Substantial cycles per season (>= 30 distinct days):")
print(counts.to_string())

gaps = []
for season, need in [("winter", ROADMAP_TARGET["min_winter_cycles"]),
                     ("post_monsoon", ROADMAP_TARGET["min_post_monsoon_cycles"])]:
    have = int(counts.get(season, 0))
    gaps.append({"season": season, "have": have, "roadmap_min": need,
                 "short_by": max(0, need - have)})
    print(f"  {season:14} have {have}, roadmap wants {need} -> "
          f"short by {max(0, need - have)}")
print("\npost_monsoon is the binding constraint: it is the crop-burning season and "
      "the source of the severe episodes this product exists to warn about. Every "
      "extreme-event metric in notebooks 06-08 is therefore measured on ONE "
      "occurrence of the phenomenon.")

,season,season_year,rows,days,mean_pm25,max_pm25,extreme_rows,is_substantial
0,monsoon,2025,377010,122,22.8,1000.0,1198,True
1,monsoon,2026,296063,101,22.7,973.0,719,True
2,post_monsoon,2025,189773,61,51.8,1000.0,5100,True
3,pre_monsoon,2025,280192,92,38.3,1000.0,2300,True
4,pre_monsoon,2026,281999,92,35.5,1870.0,1745,True
5,winter,2024,33068,10,48.1,1000.0,256,False
6,winter,2025,247147,90,60.2,1000.0,7739,True


Substantial cycles per season (>= 30 distinct days):
season
winter          1
pre_monsoon     2
monsoon         2
post_monsoon    1
  winter         have 1, roadmap wants 3 -> short by 2
  post_monsoon   have 1, roadmap wants 3 -> short by 2

post_monsoon is the binding constraint: it is the crop-burning season and the source of the severe episodes this product exists to warn about. Every extreme-event metric in notebooks 06-08 is therefore measured on ONE occurrence of the phenomenon.


In [3]:
# ============================================================================
# EXTREME-EVENT POWER - can a recall number even be estimated?
# ============================================================================
# An extreme recall of 0.70 is a claim about a rate, and a rate estimated from
# too few events has an interval wider than the gate it is being checked
# against. This cell computes the interval instead of assuming it, because
# episode count and seasonal-cycle count are different limits and only one of
# them turns out to bind here.
# ============================================================================
ext = df[df.pm25 >= 150].copy()
sev = df[df.pm25 >= 90]
print(f"Rows >= 90 ug/m3  : {len(sev):,} ({len(sev)/len(df):.2%})")
print(f"Rows >= 150 ug/m3 : {len(ext):,} ({len(ext)/len(df):.2%})")

# Count episodes, not hours: consecutive extreme hours at one station are one
# event, and a recall metric over hours double-counts a long episode.
ext = ext.sort_values(["location_id", "timestamp_utc"])
newev = (ext.groupby("location_id")["timestamp_utc"].diff() > pd.Timedelta("1h")) | \
        ext.groupby("location_id")["timestamp_utc"].diff().isna()
n_episodes = int(newev.sum())
print(f"Distinct extreme episodes: {n_episodes:,} across "
      f"{ext.location_id.nunique()} stations")

print("\nExtreme rows by season:")
print(df[df.pm25 >= 150].season.value_counts().to_string())

# The gate is measured on ONE held-out test block, not the whole dataset, so
# the interval that matters is the one for a fifth of the episodes.
p = 0.70
n_test = max(n_episodes // 5, 1)
half_width = 1.96 * np.sqrt(p * (1 - p) / n_test)
print(f"\nAll episodes      : {n_episodes:,}")
print(f"Per test block    : ~{n_test:,} (a fifth of the folds)")
print(f"95% interval on a recall of {p:.2f} in one test block: +-{half_width:.3f}")

MARGIN = 0.05      # how close to the 0.70 gate a result may land before the
                   # interval alone decides pass or fail
if half_width < MARGIN:
    print(f"\nThe episode count is ADEQUATE. At +-{half_width:.3f} the interval is "
          f"narrower than the {MARGIN:.2f} margin, so a measured recall is a real "
          "signal about this data rather than sampling noise.")
else:
    print(f"\nThe episode count is MARGINAL. At +-{half_width:.3f} the interval is "
          f"as wide as the {MARGIN:.2f} margin, so a recall near the gate cannot be "
          "called a pass or a fail.")
print("\nNote what this does and does not license. Sampling error within the "
      "observed window is small, so the recall number is precise. It says nothing "
      "about whether the number transfers to a DIFFERENT winter, and there is only "
      "one winter here - the seasonal-cycle count above is the binding limit, not "
      "the episode count.")

Rows >= 90 ug/m3  : 89,725 (5.26%)
Rows >= 150 ug/m3 : 19,057 (1.12%)
Distinct extreme episodes: 6,211 across 145 stations

Extreme rows by season:
season
winter          7995
post_monsoon    5100
pre_monsoon     4045
monsoon         1917

All episodes      : 6,211
Per test block    : ~1,242 (a fifth of the folds)
95% interval on a recall of 0.70 in one test block: +-0.025

The episode count is ADEQUATE. At +-0.025 the interval is narrower than the 0.05 margin, so a measured recall is a real signal about this data rather than sampling noise.

Note what this does and does not license. Sampling error within the observed window is small, so the recall number is precise. It says nothing about whether the number transfers to a DIFFERENT winter, and there is only one winter here - the seasonal-cycle count above is the binding limit, not the episode count.


### A ceiling in the observations

The seasonal table above reports a maximum of exactly `1000.0` in four separate seasons.
Physical maxima do not repeat to one decimal place, so this is almost certainly a reporting
or sensor ceiling rather than the true concentration.

This matters more here than it would elsewhere. Phase 7 exists to fix a −103 µg/m³ bias in
the extreme regime, and a clipped target would put a floor under that bias: a model cannot
learn a value the data refuses to record. Whether it actually matters depends entirely on
how many rows sit on the cap, so the cell below measures the share rather than stopping at
"a cap exists".

In [4]:
# ============================================================================
# CENSORING CHECK - is the extreme tail clipped?
# ============================================================================
vc = df.pm25.value_counts()
high = vc[vc.index >= 500]
# A cap is a spike relative to its NEIGHBOURS, not a large absolute count. In a
# tail this thin no value repeats often, so an absolute threshold would either
# miss the cap or fire on ordinary readings. Compare each value against the
# median frequency of other high values instead.
neighbour_median = float(high.median()) if len(high) else 1.0
spikes = high[high >= max(5 * neighbour_median, 10)].sort_values(ascending=False)
print(f"Typical repeat count for a value >= 500: {neighbour_median:.0f}")
print("Values repeating far more often than that (candidate caps):")
print(spikes.head(5).to_string() if len(spikes) else "  none")

censoring = {}
if len(spikes):
    cap = float(spikes.index[0])
    n_at_cap = int(spikes.iloc[0])
    above = int((df.pm25 > cap).sum())
    ext_n = int((df.pm25 >= 150).sum())
    share = n_at_cap / max(ext_n, 1)
    censoring = {"suspected_cap": cap, "rows_at_cap": n_at_cap,
                 "rows_above_cap": above,
                 "share_of_extreme_rows": round(share, 5),
                 "neighbour_median_count": neighbour_median,
                 "stations_affected": int(df.loc[df.pm25 == cap,
                                                 "location_id"].nunique())}
    print(f"\n  cap value        : {cap:.1f} ug/m3")
    print(f"  rows exactly at  : {n_at_cap:,}  "
          f"({n_at_cap/neighbour_median:.0f}x the typical repeat count)")
    print(f"  rows above       : {above:,}")
    print(f"  stations affected: {censoring['stations_affected']}")
    print(f"  share of all extreme (>=150) rows: {share:.3%}")

    # Whether this matters is a question of magnitude, so decide on the share
    # rather than on the existence of the spike.
    MATERIAL = 0.01
    if share >= MATERIAL:
        censoring["verdict"] = "material"
        print(f"\n  MATERIAL. Over {MATERIAL:.0%} of extreme rows sit on the cap, so "
              "the true value there is unknown and at least the cap. A model "
              "predicting above it is penalised for being right, which puts a "
              "floor under the extreme bias that notebooks 06-08 cannot tune away.")
    else:
        censoring["verdict"] = "immaterial"
        print(f"\n  IMMATERIAL. The cap is real - {n_at_cap} rows at exactly "
              f"{cap:.0f} against a typical {neighbour_median:.0f} for any other "
              f"high value, and only {above} readings above it - but it covers "
              f"{share:.3%} of extreme rows.")
        print("  It is therefore NOT an explanation for the -103 ug/m3 extreme "
              "bias. That bias has to be answered by the model, not attributed to "
              "the instrument. Recorded here so the question is closed with a "
              "number rather than left open.")
else:
    print("\nNo cap detected; the tail looks continuous.")

Typical repeat count for a value >= 500: 2
Values repeating far more often than that (candidate caps):
pm25
1000.0    49

  cap value        : 1000.0 ug/m3
  rows exactly at  : 49  (24x the typical repeat count)
  rows above       : 12
  stations affected: 7
  share of all extreme (>=150) rows: 0.257%

  IMMATERIAL. The cap is real - 49 rows at exactly 1000 against a typical 2 for any other high value, and only 12 readings above it - but it covers 0.257% of extreme rows.
  It is therefore NOT an explanation for the -103 ug/m3 extreme bias. That bias has to be answered by the model, not attributed to the instrument. Recorded here so the question is closed with a number rather than left open.


In [5]:
# ============================================================================
# STATION COVERAGE and what each analysis is supported by
# ============================================================================
per_station = (df.groupby("location_id")
                 .agg(rows=("pm25", "size"),
                      first=("timestamp_utc", "min"), last=("timestamp_utc", "max"),
                      mean_pm25=("pm25", "mean"), max_pm25=("pm25", "max")))
per_station["span_days"] = (per_station["last"] - per_station["first"]).dt.days
per_station["completeness"] = per_station.rows / (per_station.span_days * 24).clip(lower=1)
print("Per-station completeness (observations / hours in that station's own span):")
print(per_station.completeness.describe().round(3).to_string())
print(f"\nStations below 50% complete: "
      f"{int((per_station.completeness < 0.5).sum())}/{len(per_station)}")
print(f"Stations with < 90 days    : "
      f"{int((per_station.span_days < 90).sum())}/{len(per_station)}")

SUPPORT = [
    ("temporal forecasting at 1-48h", "SUPPORTED",
     "19 months of hourly data is ample for autoregressive structure"),
    ("diurnal and weekly cycles", "SUPPORTED",
     "hundreds of repetitions of each"),
    ("dispersion / stagnation physics", "SUPPORTED",
     "weather covers every hour on the grid"),
    ("crop-burning attribution", "PARTIAL",
     "one kharif season; the relationship is visible but not validated "
     "out-of-sample across years"),
    ("Diwali response", "PARTIAL",
     "two Diwalis in the window (2025-10-20, 2026-11-08) but only one with a "
     "full pre/post window inside the data"),
    ("winter episode forecasting", "NOT SUPPORTED",
     "one winter; a model tuned on it is tuned on that winter"),
    ("extreme recall >= 0.70 measured on this window", "SUPPORTED",
     f"{n_episodes:,} episodes give a +-{half_width:.3f} interval per test block"),
    ("extreme recall >= 0.70 as a claim about future winters", "NOT SUPPORTED",
     "precision within one winter is not evidence of transfer to another"),
    ("year-over-year trend or policy effect", "NOT SUPPORTED",
     "needs multiple years by definition"),
    ("spatial generalisation to unmonitored cells", "PARTIAL",
     "149 stations, mean nearest-neighbour distance ~100km, so the 1km product "
     "claim is extrapolation between sparse anchors"),
]
support = pd.DataFrame(SUPPORT, columns=["analysis", "verdict", "why"])
display(support)

Per-station completeness (observations / hours in that station's own span):
count    149.000
mean       0.842
std        0.080
min        0.447
25%        0.822
50%        0.865
75%        0.894
max        0.967

Stations below 50% complete: 1/149
Stations with < 90 days    : 0/149


,analysis,verdict,why
0,temporal forecasting at 1-48h,SUPPORTED,19 months of hourly data is ample for autoregr...
1,diurnal and weekly cycles,SUPPORTED,hundreds of repetitions of each
2,dispersion / stagnation physics,SUPPORTED,weather covers every hour on the grid
3,crop-burning attribution,PARTIAL,one kharif season; the relationship is visible...
4,Diwali response,PARTIAL,"two Diwalis in the window (2025-10-20, 2026-11..."
5,winter episode forecasting,NOT SUPPORTED,one winter; a model tuned on it is tuned on th...
6,extreme recall >= 0.70 measured on this window,SUPPORTED,"6,211 episodes give a +-0.025 interval per tes..."
7,extreme recall >= 0.70 as a claim about future...,NOT SUPPORTED,precision within one winter is not evidence of...
8,year-over-year trend or policy effect,NOT SUPPORTED,needs multiple years by definition
9,spatial generalisation to unmonitored cells,PARTIAL,"149 stations, mean nearest-neighbour distance ..."


In [6]:
# ============================================================================
# PERSIST THE AUDIT
# ============================================================================
audit = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "source_file": str(BASE_FILE.relative_to(PROJECT_ROOT)),
    "rows": int(len(df)),
    "stations": int(df.location_id.nunique()),
    "window": [str(df.timestamp_utc.min()), str(df.timestamp_utc.max())],
    "span_days": int(span_days),
    "span_years": round(float(span_years), 3),
    "roadmap_target": ROADMAP_TARGET,
    "years_short_of_minimum": round(ROADMAP_TARGET["min_years"] - float(span_years), 3),
    "seasonal_cycles": counts.to_dict(),
    "seasonal_gaps": gaps,
    "extreme_rows": int(len(ext)),
    "extreme_episodes": n_episodes,
    "extreme_recall_interval_per_test_block": round(float(half_width), 4),
    "severe_rows": int(len(sev)),
    "censoring": censoring,
    "station_completeness": {
        "median": round(float(per_station.completeness.median()), 3),
        "min": round(float(per_station.completeness.min()), 3),
        "below_50pct": int((per_station.completeness < 0.5).sum()),
    },
    "analysis_support": support.to_dict("records"),
    "blocking_source_decision": {
        "wanted": "CPCB CAAQMS 3-5 years (authoritative per the LLD)",
        "obstacle": "no public historical API; app.cpcbccr.com is a captcha-gated "
                    "JS portal, so it cannot be automated",
        "available_path": "manual portal export dropped into the folder read by "
                          "01B_cpcb_pm25_ingestion.ipynb",
        "not_attempted": "scraping the portal - it would breach the terms of use "
                         "and would be a fragile dependency for a data foundation",
    },
}
with open(OUT_DIR / "historical_coverage_audit.json", "w") as f:
    json.dump(audit, f, indent=2, default=str)

print("COVERAGE AUDIT COMPLETE\n")
print(f"  span            : {span_years:.2f} years "
      f"(roadmap minimum {ROADMAP_TARGET['min_years']})")
print(f"  winter cycles   : {counts.get('winter', 0)} "
      f"(roadmap {ROADMAP_TARGET['min_winter_cycles']})")
print(f"  post-monsoon    : {counts.get('post_monsoon', 0)} "
      f"(roadmap {ROADMAP_TARGET['min_post_monsoon_cycles']})")
print(f"  extreme episodes: {n_episodes:,}")
print(f"\n  {int((support.verdict == 'SUPPORTED').sum())} analyses supported, "
      f"{int((support.verdict == 'PARTIAL').sum())} partial, "
      f"{int((support.verdict == 'NOT SUPPORTED').sum())} not supported")
print(f"\nWrote: {OUT_DIR / 'historical_coverage_audit.json'}")
print("\nRead notebooks 06-08 against this. Where a gate fails there, this file is "
      "the first place to look for whether the model or the data is the limit.")

COVERAGE AUDIT COMPLETE

  span            : 1.55 years (roadmap minimum 3)
  winter cycles   : 1 (roadmap 3)
  post-monsoon    : 1 (roadmap 3)
  extreme episodes: 6,211

  4 analyses supported, 3 partial, 3 not supported

Wrote: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/artifacts/pm25/phase7/historical_coverage_audit.json

Read notebooks 06-08 against this. Where a gate fails there, this file is the first place to look for whether the model or the data is the limit.
